# FASE 3: PERSIAPAN & PENGGABUNGAN DATA (FEATURE ENGINEERING)
Setelah data mentahnya bersih, sekarang kita masuk ke tahap persiapan. 
Di fase ini kita akan menggabungkan titik-titik api dengan batas provinsi, menghitung jarak titik api ke sekolah, dan mencari tahu stasiun cuaca terdekatnya.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.neighbors import BallTree
import os
from IPython.display import display, Markdown

os.makedirs('../data/processed', exist_ok=True)

## 1. Gabung Titik Api dengan Peta Provinsi
Kenapa digabung? Supaya kita tahu persis tiap titik api jatuhnya di provinsi mana, dan berapa populasi penduduk di provinsi tersebut. 
Kita mengubah format data ke bentuk geospasial lalu menempelkannya ke peta Kalimantan.

In [ ]:
df_fire = pd.read_csv('../data/processed/hotspot_cleaned.csv')

gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(df_fire.longitude, df_fire.latitude),
    crs='EPSG:4326'
)

print("\n[3.2] Menggabungkan spasial hotspot ke poligon provinsi Kalimantan")
gdf_prov = gpd.read_file('../data/raw/indonesia-province-jml-penduduk.json')
gdf_prov_kalimantan = gdf_prov[gdf_prov['Propinsi'].str.contains('KALIMANTAN', case=False, na=False)].copy()
gdf_prov_kalimantan.rename(columns={'Propinsi': 'province_name', 'Jumlah Penduduk': 'population'}, inplace=True)

gdf_fire = gpd.sjoin(gdf_fire, gdf_prov_kalimantan[['geometry', 'province_name', 'population']], how='left', predicate='within')
if 'index_right' in gdf_fire.columns:
    gdf_fire.drop(columns=['index_right'], inplace=True)

unmapped = gdf_fire['province_name'].isna().sum()
display(Markdown(f"**Selesai digabung.** Ada {unmapped:,} titik api yang tidak masuk ke batas provinsi manapun (kemungkinan di laut lepas) dan dibiarkan kosong."))
display(gdf_fire.head(3))

## 2. Hitung Skala Api dan Musim Kemarau
Kita bagi tingkat keparahan api menjadi 4 level (Rendah, Menengah, Tinggi, Ekstrem) biar gampang dianalisis nanti.
Selain itu, kita butuh penanda apakah api terjadi di musim kemarau. Musim kemaraunya kita hitung otomatis pakai data curah hujan Pontianak: kalau curah hujannya di bawah 150mm sebulan, kita anggap itu bulan kemarau.

In [ ]:
gdf_fire['frp_log'] = np.log1p(gdf_fire['frp'])

q25 = gdf_fire['frp'].quantile(0.25)
q75 = gdf_fire['frp'].quantile(0.75)
q95 = gdf_fire['frp'].quantile(0.95)

def assign_frp_tier(val):
    if val < q25: return 'Rendah'
    elif val < q75: return 'Menengah'
    elif val < q95: return 'Tinggi'
    else: return 'Ekstrem'

gdf_fire['frp_tier'] = gdf_fire['frp'].apply(assign_frp_tier)

try:
    print("\n[3.3b] Mendefinisikan Musim Kemarau Berbasis Data Aktual (Curah Hujan < 150mm/bulan)")
    df_ptk = pd.read_csv('../data/raw/pontianak_weather_daily_2021_2024.csv')
    df_ptk['date'] = pd.to_datetime(df_ptk['date'], format='%d-%m-%Y')
    df_ptk['RR'] = pd.to_numeric(df_ptk['RR'], errors='coerce').fillna(0)
    monthly_rr = df_ptk.groupby([df_ptk['date'].dt.year, df_ptk['date'].dt.month])['RR'].sum().groupby(level=1).mean()
    dry_months = monthly_rr[monthly_rr < 150].index.tolist()
    print(f"     Bulan kemarau terdeteksi dari cuaca Pontianak 2021-2024 (rata-rata < 150mm/bulan): {dry_months}")
    if not dry_months:
        dry_months = [7, 8, 9, 10]
except Exception as e:
    print(f"     Gagal memuat cuaca Pontianak, fallback ke [7,8,9,10]: {e}")
    dry_months = [7, 8, 9, 10]

gdf_fire['is_dry_season'] = gdf_fire['month'].isin(dry_months)
print(f"     Total titik api selama musim kemarau (berbasis data aktual): {gdf_fire['is_dry_season'].sum():,}")

## 3. Hitung Jarak ke Sekolah Terdekat
Kita ingin tahu seberapa dekat tiap titik api dengan sekolah (karena ada anak-anak yang rentan kena asap). 
Karena titik api ada puluhan ribu dan sekolah juga banyak, menghitung jarak satu-satu bakal sangat lama. Jadi kita pakai algoritma `BallTree` (haversine) supaya komputer bisa mencari sekolah terdekat dengan sangat cepat (O(n log n) efisiensinya dibanding cara manual).

In [ ]:
fire_coords = np.radians(gdf_fire[['latitude', 'longitude']].values)

try:
    df_schools = pd.read_csv('../data/raw/complete_data.csv')
    # Batas lat/lon diselaraskan dengan bounding box Kalimantan yang didefinisikan di notebook 01
    # (lat: -4.5 s/d 4.5, lon: 108.0 s/d 119.5) — sengaja lebih lebar dari administratif agar
    # tidak ada sekolah di Kaltara/Kaltim bagian timur yang terlewat.
    df_schools = df_schools[
        (df_schools['lat'].between(-4.5, 4.5)) &
        (df_schools['long'].between(108.0, 119.5))
    ].dropna(subset=['lat', 'long'])
    
    school_coords = np.radians(df_schools[['lat', 'long']].values)
    tree = BallTree(school_coords, metric='haversine')
    dist, ind = tree.query(fire_coords, k=1)
    
    gdf_fire['dist_nearest_school_km'] = dist.flatten() * 6371
    print(f"     Rata rata jarak titik panas ke sekolah terdekat: {gdf_fire['dist_nearest_school_km'].mean():.2f} kilometer")
    
    print("\n[3.4b] Validasi densitas sekolah (OSM Under-mapping Check)")
    gdf_schools = gpd.GeoDataFrame(df_schools, geometry=gpd.points_from_xy(df_schools.long, df_schools.lat), crs='EPSG:4326')
    gdf_schools_kalim = gpd.sjoin(gdf_schools, gdf_prov_kalimantan[['geometry', 'province_name', 'population']], how='inner', predicate='within')
    school_per_prov = gdf_schools_kalim.groupby('province_name').size()
    pop_per_prov = gdf_prov_kalimantan[['province_name', 'population']].set_index('province_name')
    density_check = (school_per_prov / pop_per_prov['population'] * 100000).sort_values()
    display(Markdown("**Kepadatan sekolah per 100 ribu penduduk per provinsi:**"))
    display(density_check)
except Exception as e:
    print(f"Kesalahan saat memproses data sekolah: {e}")

## 4. Gabung dengan Data Cuaca Stasiun
Selain sekolah, kita juga pasangkan tiap titik api dengan stasiun cuaca terdekatnya. 
Tujuannya supaya kita bisa menganalisis hubungan antara titik api yang muncul dengan suhu atau curah hujan hari itu.

In [ ]:
try:
    df_stations = pd.read_csv('../data/raw/station_detail.csv')
    # Batas lat/lon diselaraskan dengan bounding box notebook 01 untuk konsistensi spatial join.
    df_stations = df_stations[
        (df_stations['latitude'].between(-4.5, 4.5)) &
        (df_stations['longitude'].between(108.0, 119.5))
    ].dropna(subset=['latitude', 'longitude'])
    
    station_coords = np.radians(df_stations[['latitude', 'longitude']].values)
    tree_stat = BallTree(station_coords, metric='haversine')
    
    dist_stat, ind_stat = tree_stat.query(fire_coords, k=1)
    gdf_fire['dist_nearest_station_km'] = dist_stat.flatten() * 6371
    gdf_fire['nearest_station_id'] = df_stations.iloc[ind_stat.flatten()]['station_id'].values
    
    df_climate = pd.read_csv('../data/raw/climate_data.csv')
    
    # Agregasi kebakaran per hari per stasiun terdekat
    daily_fires = gdf_fire.groupby(['date_local', 'nearest_station_id']).agg(
        hotspot_count=('frp', 'count'),
        max_frp=('frp', 'max'),
        mean_frp=('frp', 'mean'),
        extreme_fire_count=('frp_tier', lambda x: (x == 'Ekstrem').sum())
    ).reset_index()
    
    daily_fires.rename(columns={'date_local': 'date', 'nearest_station_id': 'station_id'}, inplace=True)
    
    if 'date' in df_climate.columns:
        df_climate['date'] = pd.to_datetime(df_climate['date'], dayfirst=True, errors='coerce')
        # SHIFT DATES BY 14 YEARS TO OVERLAP WITH HOTSPOT DATA (2024-2026) FOR DEMONSTRATION
        df_climate['date'] = df_climate['date'] + pd.DateOffset(years=14)
        df_climate['date'] = df_climate['date'].dt.strftime('%Y-%m-%d')
    elif df_climate.columns[0] not in ['Tn', 'Tx', 'Tavg']:
        df_climate['date'] = pd.to_datetime(df_climate.iloc[:, 0], errors='coerce')
        df_climate['date'] = df_climate['date'] + pd.DateOffset(years=14)
        df_climate['date'] = df_climate['date'].dt.strftime('%Y-%m-%d')
        
    df_fusion = pd.merge(df_climate, daily_fires, on=['station_id', 'date'], how='left')
    df_fusion['hotspot_count'] = df_fusion['hotspot_count'].fillna(0)
    df_fusion['extreme_fire_count'] = df_fusion['extreme_fire_count'].fillna(0)
    
    df_fusion['climate_source_year'] = pd.to_datetime(df_fusion['date']).dt.year - 14
    
    display(Markdown(f"**Data gabungan cuaca & titik api (fusion) berhasil dibuat:** {len(df_fusion):,} baris."))
    df_fusion.to_csv('../data/processed/climate_fire_fusion_SHIFTED_seasonal.csv', index=False)
    display(df_fusion.head(3))
except Exception as e:
    print(f"Kesalahan saat memproses integrasi iklim: {e}")

## 5. Simpan Hasil Gabungan
Kita buang format geometrinya karena sudah tidak diperlukan lagi, lalu kita simpan data lengkap ini ke file CSV baru. 
Data ini yang akan dipakai untuk bikin grafik di tahap selanjutnya.

In [ ]:
df_master = pd.DataFrame(gdf_fire.drop(columns='geometry'))
df_master.to_csv('../data/processed/hotspot_master.csv', index=False)
display(Markdown("**Data utama titik panas (hotspot_master) berhasil disimpan.** Siap dipakai untuk EDA!"))
display(df_master.head(3))